# Imports

In [5]:
import numpy as np
import math

# Stage 1 : Naive Unscaled Dot-Product Attention & Textbook Softmax

Full $n \times n$ Score Matrix

In [ ]:
def v1_naive_attention(Q , K , V):

    n = Q.shape[0]

    score = np.dot(Q , K.T)

    mask = np.triu(np.ones((n, n)), k=1)
    score_mask = np.where(mask == 1 , -np.inf, score)

    e_score = np.exp(score_mask)

    sum_e = np.sum(e_score , axis=1 , keepdims=True)
    P = e_score / sum_e

    O = np.dot(P , V)

    return O


# Test Data from Q1
if __name__ == "__main__":

    Q = np.array([[1, 0], 
                  [0, 2], 
                  [1, 1]], dtype=np.float32)
                  
    K = np.array([[1, 1], 
                  [2, 0], 
                  [0, 1]], dtype=np.float32)
                  
    V = np.array([[1, 0], 
                  [0, 1], 
                  [2, 3]], dtype=np.float32)

    print("=== Testing Stage 1 (Naive) ===")
    O_v1 = v1_naive_attention(Q, K, V)
    print(np.round(O_v1, 3))

=== Testing Stage 1 (Naive) ===
[[1.    0.   ]
 [0.881 0.119]
 [0.733 0.888]]


# Stage 2 : Two-Pass Safe Softmax

Variance-Scaled ($1/\sqrt{d_k}$)



In [4]:
def v2_safe_twopass_attention(Q, K, V):

    n = Q.shape[0]
    d_k = K.shape[1]

    score = np.dot(Q , K.T) / np.sqrt(d_k)

    mask = np.triu(np.ones((n, n)), k=1)
    score_mask = np.where(mask == 1 , -np.inf, score)

    m = np.max(score_mask , axis=1 , keepdims=True)

    shifted_score = score_mask - m

    e_score = np.exp(shifted_score)
    
    sum_e = np.sum(e_score , axis=1 , keepdims=True)
    P = e_score / sum_e
    
    O = np.dot(P , V)

    return O


# Test Data from Q2
if __name__ == "__main__":
    Q = np.array([[1, 0], 
                  [0, 2], 
                  [1, 1]], dtype=np.float32)
                  
    K = np.array([[1, 1], 
                  [2, 0], 
                  [0, 1]], dtype=np.float32)
                  
    V = np.array([[1, 0], 
                  [0, 1], 
                  [2, 3]], dtype=np.float32)


    scale = 1.0
    Q_large = Q * scale
    K_large = K * scale
    V_large = V

    O_v2 = v2_safe_twopass_attention(Q_large, K_large, V_large)
    print(np.round(O_v2, 3))



[[1.    0.   ]
 [0.804 0.196]
 [0.797 0.994]]


# Stage 3 : Comparing Array Passes

Stage 2 finds the row maximum and shifted denominator in two passes. Stage 3 computes both in one pass.

In [3]:
x = [2.0, 1.0, 4.0, 3.0]

# Stage 2: two passes over x
stage2_iterations = 0

m2 = -math.inf
for score in x:
    stage2_iterations += 1
    m2 = max(m2, score)

d2 = 0.0
for score in x:
    stage2_iterations += 1
    d2 += math.exp(score - m2)

print("Stage 2 passes over x:", stage2_iterations / len(x))
print("Stage 2 iterations:", stage2_iterations)
print("m2 =", m2, "d2 =", d2)

Stage 2 passes over x: 2.0
Stage 2 iterations: 8
m2 = 4.0 d2 = 1.553001792775919


Stage 3: one pass over x

In [4]:
stage3_iterations = 0
m3 = -math.inf
d3 = 0.0

for score in x:
    stage3_iterations += 1

    if score > m3:
        if m3 == -math.inf:
            d3 = 1.0
        else:
            d3 = d3 * math.exp(m3 - score) + 1.0
        m3 = score
    else:
        d3 += math.exp(score - m3)

print("Stage 3 passes over x:", stage3_iterations / len(x))
print("Stage 3 iterations:", stage3_iterations)
print("m3 =", m3, "d3 =", d3)
print("Maximum matches:", m2 == m3)
print("Denominators match:", math.isclose(d2, d3))

Stage 3 passes over x: 1.0
Stage 3 iterations: 4
m3 = 4.0 d3 = 1.553001792775919
Maximum matches: True
Denominators match: True


# Stage 4: Naive Causal Autoregressive Generation Loop

In [ ]:
def v4_autoregressive_nocache(prompt_tokens, max_new_tokens, embedding_matrix, W_q, W_k, W_v, W_out):
   
    tokens = list(prompt_tokens)
    d_k = W_q.shape[1]
    
    multiply_count = 0 
    
    for _ in range(max_new_tokens):
        embeddings = np.array([embedding_matrix[tok] for tok in tokens])
        t = embeddings.shape[0]
        
        Q = np.dot(embeddings, W_q)
        K = np.dot(embeddings, W_k)
        V = np.dot(embeddings, W_v)
        
        scores = np.dot(Q, K.T) / np.sqrt(d_k)
        multiply_count += 2 * (t * t * d_k)
        
        mask = np.triu(np.ones((t, t)), k=1)
        scores_masked = np.where(mask == 1, -np.inf, scores)
        
        m = np.max(scores_masked, axis=1, keepdims=True)
        exp_scores = np.exp(scores_masked - m)
        P = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        
        contexts = np.dot(P, V)
        
        last_context = contexts[-1]
        logits = np.dot(last_context, W_out)
        next_token = int(np.argmax(logits))
        
        tokens.append(next_token)
        
    return tokens, multiply_count


if __name__ == "__main__":

    embedding_matrix = {
        0: [1.0, 0.0],
        1: [0.0, 1.0],
        2: [1.0, 1.0],
    }
    identity = np.array([[1.0, 0.0], [0.0, 1.0]])
    
    W_out = np.array([[0.1, 0.9, 0.2], 
                      [0.1, 0.9, 0.2]])

    result_tokens, total_multiplies = v4_autoregressive_nocache(
        prompt_tokens=[0],
        max_new_tokens=3,
        embedding_matrix=embedding_matrix,
        W_q=identity,
        W_k=identity,
        W_v=identity,
        W_out=W_out
    )

    print("Generated tokens:", result_tokens)
    print("Total QK Multiplications:", total_multiplies)

Generated tokens: [0, 1, 1, 1]
Total QK Multiplications: 28


# Stage 5 theoretical decode operation 

counts, T = 3000, d = 64 The problem uses N4 = T(T+1)(2T+1)d/3 and N5 = T(T+1)d Two cache arrays each store T×d float numbers.

In [1]:
T, d = 3000, 64

N4 = T * (T + 1) * (2 * T + 1) // 3 * d
N5 = T * (T + 1) * d

print("Stage 4 multiplications:", f"{N4:,}")
print("Stage 5 multiplications:", f"{N5:,}")
print("Theoretical ratio:", N4 / N5)
print("K+V cached floats:", 2 * T * d)
print("float32 bytes:", 2 * T * d * 4)

Stage 4 multiplications: 1,152,576,064,000
Stage 5 multiplications: 576,192,000
Theoretical ratio: 2000.3333333333333
K+V cached floats: 384000
float32 bytes: 1536000
